In [1]:
import importlib
import src.data
importlib.reload(src.data)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.preprocessing_data import prepare_training_data
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

In [2]:
from src.data import load_raw_tables, prepare_delivery_events, build_master_df, add_engineering_features

In [3]:
base_path = r"C:\Users\jinfe\.cache\kagglehub\datasets\yogape\logistics-operations-database\versions\1"
tables = load_raw_tables(base_path)
delivery_events = prepare_delivery_events(tables["delivery_events"])
df = build_master_df(
    delivery_events,
    tables["trips"],
    tables["loads"],
    tables["drivers"],
    tables["trucks"],
    tables["routes"],
    tables["customers"]
)
df = add_engineering_features(df)

In [4]:
df

,event_id,load_id,trip_id,event_type,facility_id,scheduled_datetime,actual_datetime,detention_minutes,on_time_flag,location_city,...,typical_transit_days,customer_type,month,truck_age_at_event,distance_deviation,is_home_terminal,detention_risk_index,efficiency_ratio,revenue_density,is_winter
0,EVT00000001,LOAD00000001,TRIP00000001,Pickup,FAC00034,2022-01-01 18:00:00.000000,2022-01-01 20:58:55.918185,0,False,Houston,...,2,Spot,1,7.0,43,0,1051.0,0.893750,0.158779,1
1,EVT00000002,LOAD00000001,TRIP00000001,Delivery,FAC00046,2022-01-02 23:10:55.918185,2022-01-02 21:30:22.142060,230,True,Detroit,...,2,Spot,1,7.0,43,0,1897.0,0.893750,0.158779,1
2,EVT00000003,LOAD00000002,TRIP00000002,Pickup,FAC00015,2022-01-01 18:00:00.000000,2022-01-01 17:37:26.608430,62,True,Kansas City,...,1,Spot,1,7.0,-4,0,1457.0,0.687500,0.044106,1
3,EVT00000004,LOAD00000002,TRIP00000002,Delivery,FAC00050,2022-01-02 02:13:26.608430,2022-01-02 05:55:46.238257,67,False,Indianapolis,...,1,Spot,1,7.0,-4,0,2246.0,0.687500,0.044106,1
4,EVT00000005,LOAD00000003,TRIP00000003,Pickup,FAC00001,2022-01-01 08:00:00.000000,2022-01-01 07:28:02.169634,83,True,Columbus,...,4,Spot,1,4.0,177,0,2166.0,1.480000,0.201464,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
170815,EVT00170816,LOAD00085408,TRIP00085408,Delivery,FAC00045,2025-01-01 17:57:42.158981,2025-01-01 15:19:00.254154,126,False,Portland,...,2,Contract,1,9.0,4,0,2526.0,0.707000,0.314411,1
170816,EVT00170817,LOAD00085409,TRIP00085409,Pickup,FAC00025,2024-12-31 13:00:00.000000,2024-12-31 16:37:42.056974,13,False,Philadelphia,...,4,Dedicated,12,8.0,29,0,2123.0,0.737778,0.184528,1
170817,EVT00170818,LOAD00085409,TRIP00085409,Delivery,FAC00001,2025-01-02 16:31:42.056974,2025-01-02 21:15:48.779380,81,False,Seattle,...,4,Dedicated,1,9.0,29,0,1160.0,0.664000,0.184528,1
170818,EVT00170819,LOAD00085410,TRIP00085410,Pickup,FAC00027,2024-12-31 06:00:00.000000,2024-12-31 05:30:59.579843,48,True,Dallas,...,2,Dedicated,12,9.0,40,0,1319.0,0.682000,0.388001,1


In [5]:
# Removing rows with missing values in 'CustomerID' and 'Description' columns
df = df.dropna(subset=['is_home_terminal', 'years_experience','efficiency_ratio','truck_age_at_event'])
df.head()

,event_id,load_id,trip_id,event_type,facility_id,scheduled_datetime,actual_datetime,detention_minutes,on_time_flag,location_city,...,typical_transit_days,customer_type,month,truck_age_at_event,distance_deviation,is_home_terminal,detention_risk_index,efficiency_ratio,revenue_density,is_winter
0,EVT00000001,LOAD00000001,TRIP00000001,Pickup,FAC00034,2022-01-01 18:00:00.000000,2022-01-01 20:58:55.918185,0,False,Houston,...,2,Spot,1,7.0,43,0,1051.0,0.89375,0.158779,1
1,EVT00000002,LOAD00000001,TRIP00000001,Delivery,FAC00046,2022-01-02 23:10:55.918185,2022-01-02 21:30:22.142060,230,True,Detroit,...,2,Spot,1,7.0,43,0,1897.0,0.89375,0.158779,1
2,EVT00000003,LOAD00000002,TRIP00000002,Pickup,FAC00015,2022-01-01 18:00:00.000000,2022-01-01 17:37:26.608430,62,True,Kansas City,...,1,Spot,1,7.0,-4,0,1457.0,0.68750,0.044106,1
3,EVT00000004,LOAD00000002,TRIP00000002,Delivery,FAC00050,2022-01-02 02:13:26.608430,2022-01-02 05:55:46.238257,67,False,Indianapolis,...,1,Spot,1,7.0,-4,0,2246.0,0.68750,0.044106,1
4,EVT00000005,LOAD00000003,TRIP00000003,Pickup,FAC00001,2022-01-01 08:00:00.000000,2022-01-01 07:28:02.169634,83,True,Columbus,...,4,Spot,1,4.0,177,0,2166.0,1.48000,0.201464,1


In [6]:
# Final Column Selection (The Optimized List)
final_columns = [
    'on_time_flag', 
    'location_state'
    'facility_congestion_score', 
    'facility_detention_avg',
    'hour_of_day', 
    'day_of_week', 
    'event_type',
    'booking_type',
    'weight_lbs', 
    'pieces',
    'revenue',
    'load_type',
    'years_experience', 
    'truck_age_at_event',
    'average_mpg', 
    'idle_time_hours',
    'distance_deviation',
    'is_home_terminal', 
    'detention_risk_index', 
    'efficiency_ratio', 
    'revenue_density',
    'is_winter'
]

In [7]:
# Separate Features and Target
categorical_cols = ['day_of_week', 'event_type', 'booking_type', 'load_type','location_state']
numeric_cols = [
    'facility_congestion_score', 'facility_detention_avg', 'hour_of_day', 
    'weight_lbs', 'pieces', 'revenue', 'years_experience', 
    'truck_age_at_event', 'average_mpg', 'idle_time_hours', 'distance_deviation',
    'is_home_terminal', 'detention_risk_index', 'efficiency_ratio', 'revenue_density','is_winter'
]
target = 'on_time_flag'

In [8]:
# Create df for model training
df_model = df[categorical_cols + numeric_cols + [target]].copy()

In [9]:
# ONE-HOT ENCODING to convertstext categories into 0/1 columns
df_encoded = pd.get_dummies(df_model, columns=categorical_cols, drop_first=True)

dummy_cols = [col for col in df_encoded.columns if any(col.startswith(cat + "_") for cat in categorical_cols)]

correlation_matrix_c = df_encoded[dummy_cols + [target]].corr()

we identified several instances of Multicollinearity—where features were too closely related to each other. To ensure our Logistic Regression model remained stable and interpretable, we performed Feature Selection. We prioritized our 'Smart' engineered features, like the Detention Risk Index, over raw metrics because they provided a more condensed and powerful signal, reducing redundancy and improving model efficiency."
Model Confusion: When two variables are highly correlated (e.g., facility_congestion_score and detention_risk_index), Logistic Regression struggles to figure out which one is actually causing the delay. This makes your "Odds Ratios" and coefficients unreliable.

Overfitting: Including redundant information can lead to a model that is "noisier" and harder to generalize to new data.

Redundancy: If pieces and weight_lbs have a correlation of, say, 0.95, they are essentially telling the same story. Keeping both adds computational cost without adding new information.

The "Rules of Thumb" for your PresentationCorrelation ValueActionReason0.0 to 0.7Keep BothLow to moderate correlation; both provide unique signals.0.7 to 0.9InvestigateHigh correlation. Check if one is a derivative of the other.0.9 to 1.0Remove OneExtreme redundancy. Keeping both will likely hurt your Logistic Regression coefficients.

using PCA (Principal Component Analysis) is a highly effective way to handle the multicollinearity you saw in your heatmaps!
Why use PCA for your project?
Solves Multicollinearity: PCA components are mathematically guaranteed to be independent (orthogonal). This perfectly fixes the issues Logistic Regression has with correlated features.

Reduces Noise: It keeps the "signal" and ignores the random "noise" in the data.

Simplifies the Model: Instead of 20+ features, you might find that 5 or 10 Principal Components capture 95% of the information.

3. The "Downside" for your Presentation
While PCA improves performance and handles correlation, it kills interpretability.

Before PCA: You can say, "If weight increases, the load is more likely to be late."

After PCA: You have to say, "If Principal Component 1 increases, the load is more likely to be late." (But you don't know exactly what PC1 represents—it's a mix of weight, pieces, and revenue).

4. Presentation Speech (The "Pro" Strategy)
If you use PCA, explain it as a solution to the heatmaps you shared:

Speech Script:
"Observing the high multicollinearity in our heatmap, we decided to implement PCA (Principal Component Analysis). This allowed us to transform our correlated features into a set of linearly independent components. While this makes individual feature interpretation more complex, it significantly stabilized our Logistic Regression model and allowed us to capture 95% of the data's variance while reducing the dimensionality of the problem. This ensured the model focused on the 'signal' rather than the redundant 'noise' between similar variables like weight and pieces."

In [10]:
from sklearn.decomposition import PCA


In [11]:
X_train_scaled, X_test_scaled, y_train, y_test, imputer, scaler = prepare_training_data(
    df_encoded, target
)

In [12]:
# 2. Apply PCA
# We start by not limiting components to see how much variance they explain
pca = PCA()
X_pca = pca.fit_transform(X_train_scaled)

# 3. Determine how many components to keep (The "Scree Plot")
exp_var_cumul = np.cumsum(pca.explained_variance_ratio_)


In [13]:
def apply_pca(X_train_scaled, X_test_scaled, n_components=0.95):
    pca = PCA(n_components=n_components)

    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)

    return X_train_pca, X_test_pca, pca


"We split the data before scaling to prevent Data Leakage. By fitting the scaler only on the training data, we ensure that the model has no knowledge of the mean or standard deviation of the test set, which simulates a real-world scenario where the model sees completely new, unseen data."